# Your first prediction: stock 1

**No experience needed.** If you have never written code or done any statistics, this notebook is for you. Read the text, run the code, and by the end you will have a real quote for round 1 of the hackathon.

### The plan

| Part | What we do | In everyday words |
|---|---|---|
| 1 | Load the data | Open the spreadsheet |
| 2 | Look at the data | Get to know it |
| 3 | Train a model | Find the pattern |
| 4 | Check the model | See how often it's wrong, and by how much |
| 5 | Predict stock 1 | Use the pattern on the new row |
| 6 | Make a quote | Turn the prediction into a bid and an ask |

It takes about 30–45 minutes if you read everything.

> **About AI tools.** You may use AI assistants during the hackathon, and they're great for writing code. But make sure you understand what your code does. When the game is live you'll need to make quick decisions about quotes and trades, and that's much easier if you know where your numbers come from and when to doubt them.

## Before you start: how a notebook works

This document is a **Jupyter notebook**. It's made of boxes called **cells**:

* **Text cells**, like this one. Just read them.
* **Code cells**: grey boxes containing Python code. Click on one and press **Shift + Enter** to *run* it. Its result appears just below it.

**Three golden rules**

1. **Run the cells in order, top to bottom.** Later cells use things that earlier cells created.
2. **If something gets weird, start again:** menu **Kernel → Restart Kernel and Run All Cells** (in VS Code: **Restart**, then **Run All**).
3. **You can't break anything.** Edit the code, experiment, re-run. The data files are never changed.

Try it now: click on the cell below and press **Shift + Enter**.

In [ ]:
print("Hello! The notebook is working.")

You should see `Hello! The notebook is working.` underneath. 🎉

### Python in two minutes

You only need four ideas to follow this notebook.

**1. Variables store things.** `x = 5` puts the number 5 in a box called `x`. Later, `x` means 5.

**2. `print(...)` shows things on screen.**

**3. Lines starting with `#` are comments.** They're notes for humans; Python ignores them.

**4. Dots and brackets.** `something.do_this()` asks `something` to do an action (like `table.head()`, "show the top of the table"). `table["price"]` picks out the column called `price`.

Run this little example:

In [ ]:
apples = 3            # a variable called apples holds the number 3
price_each = 0.5      # another variable
total = apples * price_each   # * means multiply
print("Total cost:", total)

### If you see a red error message, don't panic

Errors are normal, even for experts. The **last line** of the error tells you what went wrong. The usual ones:

| Error | What it usually means | Fix |
|---|---|---|
| `ModuleNotFoundError` | A library isn't installed. | In a terminal, in the repository folder, run `pip install -r requirements.txt`, then restart the notebook. |
| `FileNotFoundError` | Python can't find the data file. | Make sure the notebook is inside the `notebooks` folder of the repository and `datasets` is next to it. |
| `NameError: name '...' is not defined` | You skipped a cell. | Run all the cells above, in order (or Restart and Run All). |

---
# Part 1: Load the data

### Step 1.1: Load the tools

Python has free add-ons called **libraries**. Each one is a toolbox someone else built. We use four:

| Library | Nickname | Think of it as |
|---|---|---|
| **pandas** | `pd` | Excel inside Python: tables of data. |
| **NumPy** | `np` | A calculator for whole columns of numbers at once. |
| **Matplotlib** | `plt` | A chart maker. |
| **scikit-learn** | `sklearn` | A shop of ready-made prediction models. |

`import pandas as pd` means "open the pandas toolbox, and let me call it `pd`". Everyone uses these same nicknames, so code you find online will look familiar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

print("All tools loaded.")

### Step 1.2: Open the data files

The data are **CSV files** ("comma-separated values"). A CSV file is just a table saved as text: one line per row, with commas between the values. You could open it in Excel.

For every stock there are two files:

* **`stock_1_train.csv`, the training data.** Lots of past examples, each **with** its price. We learn from these.
* **`stock_1_test.csv`, the test row.** **One** example **without** its price. This is what you have to price in round 1. Only the organisers know the answer, and they reveal it at the end of the round.

`pd.read_csv("file name")` opens a CSV file as a table. pandas calls a table a **DataFrame**.

In [ ]:
# The data live in the "datasets" folder, one level up from this notebook ("..").
DATA_DIR = Path("../datasets")
if not DATA_DIR.exists():          # in case the notebook was started from the main folder instead
    DATA_DIR = Path("datasets")

train = pd.read_csv(DATA_DIR / "stock_1_train.csv")   # the examples we learn from
test = pd.read_csv(DATA_DIR / "stock_1_test.csv")     # the one row we must price

print("Data loaded.")

### Step 1.3: Rows and columns

A table has **rows** (going across) and **columns** (going down).

* **One row = one example.** Here, one past moment when we recorded some measurements and the price that followed.
* **One column = one kind of information**, recorded for every example.

`.shape` tells us the size: (number of rows, number of columns).

In [ ]:
print("Training data:", train.shape[0], "rows and", train.shape[1], "columns")
print("Test data:    ", test.shape[0], "row  and", test.shape[1], "columns")

That's 20,000 examples to learn from, and 1 to predict.

`.head()` shows the first 5 rows. The numbers in bold on the far left (0, 1, 2, ...) are just pandas counting the rows; they aren't part of the data.

In [ ]:
train.head()

### What each column means

| Column | What it is |
|---|---|
| `uid` | A unique code for each row, like a name tag. It labels the row, it doesn't measure anything. |
| `x1` to `x6` | **Features**: measurements that might help predict the price. Their real names are hidden, so we just call them `x1` ... `x6`. |
| `price` | **The target**: the thing we want to predict, in pounds (£). Only the training file has it. |

Two words you'll hear all the time:

* **Feature** = an input (a clue).
* **Target** = the output (the answer we want).

A **model** is a recipe that turns the clues into a guess of the answer.

Now the test row. It has the same clues but no price (and an `id` column that just names the stock):

In [ ]:
test

### Step 1.4: Quick health check

Before using data, check two things.

**Are there empty cells?** `train.isna().sum()` counts the empty cells in each column. We want zeros.

In [ ]:
train.isna().sum()

All zeros: no empty cells in stock 1. (Not every stock will be this tidy.)

**What are typical values?** `.describe()` gives a quick summary of each column:

* `mean`: the average.
* `std`: the **standard deviation**, i.e. how spread out the values are. A typical value is roughly one `std` away from the average.
* `min` and `max`: the smallest and largest values.
* `50%`: the **median**, the middle value if you sorted them all. (`25%` and `75%` are the quarter-way and three-quarter-way points.)

In [ ]:
train.describe().round(2)

For example, the `price` column has an average of about £125, and most prices are within about £15 of that.

> **✅ Checkpoint.** You've loaded the data, you know that each row is an example and each column a piece of information, and you know `price` is what we want to predict.

---
# Part 2: Look at the data

Pictures beat tables. Always plot your data before modelling it: you'll spot patterns and problems straight away.

### Step 2.1: What do prices look like?

A **histogram** sorts the prices into buckets and draws a bar for each bucket. Taller bar = more rows with prices in that range.

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.hist(train["price"], bins=60, color="steelblue", edgecolor="white")
plt.xlabel("price (£)")
plt.ylabel("how many rows")
plt.title("Stock 1: how prices are spread out")
plt.show()

A single hump, centred around £125. Nothing strange.

### Step 2.2: Does each clue affect the price?

A **scatter plot** draws one dot per row: the clue across the bottom, the price up the side. Look at the overall shape of the cloud:

| The cloud... | It means... |
|---|---|
| goes **up** to the right ↗ | bigger values of this clue come with **higher** prices |
| goes **down** to the right ↘ | bigger values come with **lower** prices |
| is a **flat blob** | this clue, on its own, says little about the price |

20,000 dots would be too crowded, so we plot a random 2,000. (`random_state=0` just means "pick the same random rows every time".)

In [ ]:
features = ["x1", "x2", "x3", "x4", "x5", "x6"]
sample = train.sample(2000, random_state=0)

fig, axes = plt.subplots(2, 3, figsize=(12, 6.5), sharey=True)
for ax, col in zip(axes.flat, features):        # one small plot per clue
    ax.scatter(sample[col], sample["price"], s=4, alpha=0.4)
    ax.set_xlabel(col)
axes[0, 0].set_ylabel("price (£)")
axes[1, 0].set_ylabel("price (£)")
fig.suptitle("Stock 1: price against each clue")
plt.tight_layout()
plt.show()

**What do you see?** `x1`, `x3` and `x4` slope up. `x2` slopes clearly down. `x5` and `x6` look like flat blobs. And the slopes look roughly like **straight lines**, which is good news for the model we're about to use.

> **✅ Checkpoint.** Some clues push the price up, some push it down, and some don't seem to matter. The patterns look straight.

---
# Part 3: Train a model (linear regression)

### Step 3.1: The idea, with one clue

**Linear regression** is the simplest prediction model there is: it draws **the best straight line** through the cloud of dots.

"Best" means: the line is placed so the dots are, on average, as close to it as possible. (Technically, it makes the sum of the *squared* vertical distances as small as possible. That's why it's also called **least squares**.)

Here's the best line for `x3` alone:

In [ ]:
one_clue_model = LinearRegression()
one_clue_model.fit(train[["x3"]], train["price"])     # "fit" = find the best line

start = one_clue_model.intercept_      # where the line crosses x3 = 0
slope = one_clue_model.coef_[0]        # how steep the line is

xs = np.linspace(-4, 4, 50)
plt.figure(figsize=(7, 3.5))
plt.scatter(sample["x3"], sample["price"], s=4, alpha=0.4, label="data")
plt.plot(xs, start + slope * xs, color="crimson", linewidth=3, label="best straight line")
plt.xlabel("x3"); plt.ylabel("price (£)"); plt.legend()
plt.show()

print(f"The line says: price ≈ {start:.1f} + {slope:.2f} × x3")

Read it like this: **every time `x3` goes up by 1, the predicted price goes up by about £6.**

The line follows the trend, but the dots are still spread widely around it. That's because the *other* clues also matter. So let's use them all.

### Step 3.2: Using all the clues

With several clues, linear regression simply adds up one piece per clue:

$$\text{predicted price} = \text{start} + w_1 \times x_1 + w_2 \times x_2 + \dots + w_6 \times x_6$$

* **start** (called the **intercept**) is the price when every clue is 0.
* Each **w** is a **weight** (called a **coefficient**): how much that clue moves the price.

The computer's job is to find the best start and weights from the 20,000 examples.

### Step 3.3: Separate the clues from the answer

By tradition, the table of clues is called **`X`** and the column of answers is called **`y`**. We leave out `uid` (it's a label, not a measurement) and of course `price` (that's the answer!).

In [ ]:
clue_columns = ["x1", "x2", "x3", "x4", "x5", "x6"]

X = train[clue_columns]     # the clues
y = train["price"]          # the answers

### Step 3.4: Keep some data aside for a fair test

Imagine a student who memorises last year's exam paper and then gets marked on that same paper. They'd look brilliant, but we'd learn nothing about how they'll do on a *new* exam.

Models are the same. So we split the training data in two:

* **80% for learning** (the model studies these),
* **20% for checking** (kept hidden from the model, like a mock exam).

The checking part tells us honestly how the model does on rows it has never seen, which is exactly the situation of the test row.

In [ ]:
X_learn, X_check, y_learn, y_check = train_test_split(X, y, test_size=0.2, random_state=0)

print("Rows for learning:", len(X_learn))
print("Rows for checking:", len(X_check))

### Step 3.5: Train it

Every model in scikit-learn works in the same three steps:

1. **Create** it: `model = LinearRegression()`
2. **Train** it: `model.fit(clues, answers)`, which is where it finds the start and the weights
3. **Use** it: `model.predict(new_clues)`

In [ ]:
model = LinearRegression()
model.fit(X_learn, y_learn)

print(f"start (intercept) = {model.intercept_:.2f}")
pd.DataFrame({"clue": clue_columns, "weight": model.coef_.round(2)})

**How to read the weights**

* `x3` has weight about **+6**: when `x3` goes up by 1, the price goes up by about £6.
* `x2` has weight about **−0.8**: when `x2` goes up by 1, the price goes *down* by about 80p.
* `x5` and `x6` have weights close to **0**: the model found they don't matter, just as the scatter plots suggested.

> **✅ Checkpoint.** You've trained your first model! It's just a starting number plus a weight for each clue.

---
# Part 4: How good is the model?

Now the mock exam: predict the 20% of rows the model has never seen, and compare with their real prices.

For each row, the **error** is: *real price − predicted price*. We summarise all the errors with two numbers:

* **Average error (MAE)**: on average, how many pounds off are we?
* **RMSE**: similar, but big misses count extra. **This is the number we'll use to size our quote.**

In [ ]:
predicted = model.predict(X_check)
errors = y_check - predicted

mae = np.mean(np.abs(errors))             # average size of the errors
rmse = np.sqrt(np.mean(errors ** 2))      # root-mean-square error

print(f"On average the model is £{mae:.2f} off.")
print(f"RMSE: £{rmse:.2f}")

Let's see the errors. On the left, each dot is one row: if the model were perfect, every dot would sit on the red line. On the right, a histogram of the errors.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))

left.scatter(y_check, predicted, s=3, alpha=0.3)
left.plot([70, 180], [70, 180], color="crimson")
left.set_xlabel("real price (£)"); left.set_ylabel("predicted price (£)")
left.set_title("Predicted vs real")

right.hist(errors, bins=60, color="steelblue", edgecolor="white")
right.set_xlabel("error = real − predicted (£)"); right.set_ylabel("how many rows")
right.set_title("Errors")

plt.tight_layout()
plt.show()

The dots hug the red line, and the errors form a neat bell shape centred on zero, mostly within about ±£6. That's a good model for this stock. The remaining errors look like pure randomness that no model could predict.

> **✅ Checkpoint.** The model is usually about £2–3 off, and rarely more than £6–9. Remember this: it's the key to choosing your quote.

---
# Part 5: Predict stock 1

### Step 5.1: Is the test row "normal"?

A model is only trustworthy for rows that look like the ones it learned from. So let's compare the test row's values with the smallest and largest values in the training data.

In [ ]:
check = pd.DataFrame({
    "test row": test[clue_columns].iloc[0],
    "smallest in training": train[clue_columns].min(),
    "largest in training": train[clue_columns].max(),
})
check

Every test value lies comfortably between the smallest and largest training values. Good: no surprises.

### Step 5.2: Retrain on everything, then predict

We set 20% aside only to measure the model. Now that we trust it, we train it again on **all** 20,000 rows (more examples, slightly better weights) and predict the test row.

In [ ]:
final_model = LinearRegression()
final_model.fit(X, y)                  # all 20,000 rows

prediction = final_model.predict(test[clue_columns])[0]    # [0] takes the single number out
print(f"Predicted price of stock 1: £{prediction:.2f}")

### Step 5.3: No magic, just arithmetic

To prove there's nothing mysterious, let's redo that prediction by hand: start + weight × value, for each clue.

In [ ]:
total = final_model.intercept_
print(f"start{'':22} {total:8.2f}")
for clue, weight in zip(clue_columns, final_model.coef_):
    value = test[clue].iloc[0]
    total += weight * value
    print(f"+ {weight:6.2f} × {clue} ({value:7.2f})   {weight * value:+8.2f}")
print(f"{'= prediction':28} {total:8.2f}")

---
# Part 6: Turn the prediction into a quote

### Step 6.1: Quick reminder of the game

In each round you submit two prices:

* **Bid** = the price at which you'd **buy**.
* **Ask** = the price at which you'd **sell**. It must be higher than your bid.
* **Spread** = ask − bid.

The team with the **smallest spread** becomes the **market maker**, and every other team trades with it: they buy at its ask or sell at its bid. If its prices are wrong, it loses money to the teams that guessed better.

So the question is: **how wide should my quote be?**

### Step 6.2: Use the model's typical error

The real price will almost never be exactly our prediction. In Part 4 we measured how far off the model usually is: the **RMSE**. Because the errors were bell-shaped, a handy rule of thumb is:

| Quote | How often the real price lands inside it |
|---|---|
| prediction ± ½ RMSE | about 4 times in 10 |
| prediction ± 1 RMSE | about 7 times in 10 |
| prediction ± 2 RMSE | about 19 times in 20 |

* **Narrow quote**: more likely to win the market-maker seat, but more likely to be wrong.
* **Wide quote**: safer, but someone else probably becomes the market maker.

Here are the three options for stock 1:

In [ ]:
for k in [0.5, 1, 2]:
    bid = prediction - k * rmse
    ask = prediction + k * rmse
    print(f"prediction ± {k} RMSE:   bid £{bid:.2f}   ask £{ask:.2f}   (spread £{ask - bid:.2f})")

### Step 6.3: Your quote

**This is your decision.** Change `half_width` below to choose how wide to quote, then run the cell. Prices are rounded to the penny, as the game requires.

In [ ]:
half_width = 1 * rmse          # ← try 0.5 * rmse, 2 * rmse, or any amount in pounds, e.g. 4.0

my_bid = round(prediction - half_width, 2)
my_ask = round(prediction + half_width, 2)

print("=" * 34)
print(f"  YOUR QUOTE FOR STOCK 1")
print(f"  Bid: £{my_bid:.2f}")
print(f"  Ask: £{my_ask:.2f}")
print(f"  Spread: £{my_ask - my_bid:.2f}")
print("=" * 34)

### Step 6.4: If someone else becomes the market maker

When the trading phase starts, you'll see the market maker's bid and ask. Compare them with **your** prediction:

* Your prediction is **above their ask**? → **Buy** (you think they're selling too cheap).
* Your prediction is **below their bid**? → **Sell** (you think they're paying too much).
* Your prediction is **in between**? → No clear advantage. Trade the minimum (10 shares) in the direction you lean. (If you don't trade at all, the game makes a random 10-share trade for you.)

Type the market maker's prices into the cell below to get a suggestion.

In [ ]:
mm_bid = 130.00     # ← type the market maker's bid here
mm_ask = 133.00     # ← type the market maker's ask here

if prediction > mm_ask:
    print(f"BUY at £{mm_ask:.2f}. You expect to make about £{prediction - mm_ask:.2f} per share.")
elif prediction < mm_bid:
    print(f"SELL at £{mm_bid:.2f}. You expect to make about £{mm_bid - prediction:.2f} per share.")
else:
    print("No clear edge. Trade the minimum, 10 shares, in the direction your prediction leans.")

**How many shares?** More shares = more profit if you're right, more loss if you're wrong. Only trade big when the gap between your prediction and their price is large compared with your RMSE.

> **✅ Checkpoint.** You have a prediction, a quote, and a plan for trading. You're ready for round 1!

---
# Everything in one cell

Here is the whole recipe from this notebook in a single cell, so you can see it at a glance.

In [ ]:
STOCK = 1

train = pd.read_csv(DATA_DIR / f"stock_{STOCK}_train.csv")
test = pd.read_csv(DATA_DIR / f"stock_{STOCK}_test.csv")
clues = ["x1", "x2", "x3", "x4", "x5", "x6"]

# 1) measure the typical error on rows the model hasn't seen
X_learn, X_check, y_learn, y_check = train_test_split(train[clues], train["price"], test_size=0.2, random_state=0)
check_model = LinearRegression().fit(X_learn, y_learn)
rmse = np.sqrt(np.mean((y_check - check_model.predict(X_check)) ** 2))

# 2) retrain on everything and predict the test row
prediction = LinearRegression().fit(train[clues], train["price"]).predict(test[clues])[0]

# 3) quote
print(f"prediction £{prediction:.2f}, typical error £{rmse:.2f}")
print(f"quote: bid £{prediction - rmse:.2f} / ask £{prediction + rmse:.2f}")

---
# What next?

**Try the other stocks.** But be warned: **this exact recipe won't simply work on the others.** The stocks are all different: some have many more columns, some have text columns or empty cells, some are much noisier, and some hide surprises. You'll need to adapt, and that's the fun part.

Five habits that will help on every stock:

1. **Look before you model.** Open the data, make plots, check for empty cells and anything odd.
2. **Test honestly** on rows the model hasn't seen.
3. **Compare the test row with the training data.** Is it a normal row?
4. **Think about how wrong you could be**, not just your best guess. Your quote is about both.
5. **Understand your tools.** Whether you write the code yourself or with an AI assistant, make sure you can explain what it does. If two sensible approaches give very different answers, find out why before you quote.

Good luck! 🍀

---

### Glossary

| Word | Meaning |
|---|---|
| **DataFrame** | A table in pandas. |
| **Row** | One example. |
| **Column** | One kind of information. |
| **Feature / clue** | An input column used to predict. |
| **Target** | The column we want to predict (`price`). |
| **Training data** | Examples with known answers that the model learns from. |
| **Test row** | The example we must predict; its answer is hidden. |
| **Linear regression** | A model that predicts with a starting number plus a weight times each clue (a "best straight line"). |
| **Intercept / coefficient** | The starting number / the weight on a clue. |
| **Error** | Real − predicted. |
| **MAE / RMSE** | Two ways of measuring the typical size of the errors, in £. |
| **Bid / ask / spread** | Price you'd buy at / price you'd sell at / the gap between them. |
| **Market maker** | The team with the smallest spread; everyone else trades with it. |